# Notebook 04 — CP01: Visualization and Biological Interpretation

**Module:** 20 — Capstone Projects  
**Tier:** 1 — Master  
**Notebook:** 4 of 12  
**Time estimate:** 75 minutes

> This notebook produces the publication-quality figures for CP01 and
> writes the Results paragraph that accompanies them.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)

# ---- Regenerate data (same seed as NB02/NB03) ----
N_GENES = 5000; N_SAMPLES = 6; N_DE = 300
mu_base    = rng.lognormal(3.0, 1.5, N_GENES)
dispersion = rng.exponential(0.1, N_GENES) + 0.01
r = 1.0 / dispersion[:, None]; p = 1.0 / (1.0 + dispersion[:, None] * mu_base[:, None])
counts = rng.negative_binomial(r, p, size=(N_GENES, N_SAMPLES)).astype(float)
de_idx = rng.choice(N_GENES, N_DE, replace=False)
up_idx, dn_idx = de_idx[:N_DE//2], de_idx[N_DE//2:]
fold_changes = rng.uniform(1.5, 4.0, N_DE)
counts[up_idx, 3:] = (counts[up_idx, 3:] * fold_changes[:N_DE//2, None]).astype(int)
counts[dn_idx, 3:] = np.maximum(1, (counts[dn_idx, 3:] / fold_changes[N_DE//2:, None]).astype(int))
counts = np.clip(counts, 0, None)
gene_names = [f'GENE_{i:04d}' for i in range(N_GENES)]

# Size factors + normalization
log_counts   = np.log(counts + 1e-9)
log_geo_mean = log_counts.mean(axis=1)
mask         = (counts > 0).all(axis=1)
log_ratios   = log_counts - log_geo_mean[:, None]
size_factors = np.exp(np.median(log_ratios[mask], axis=0))
norm_counts  = counts / size_factors[None, :]

PSEUDOCOUNT = 1.0
log2_ctrl = np.log2(norm_counts[:, :3].mean(axis=1) + PSEUDOCOUNT)
log2_trt  = np.log2(norm_counts[:, 3:].mean(axis=1) + PSEUDOCOUNT)
log2fc    = log2_trt - log2_ctrl
log2_norm = np.log2(norm_counts + PSEUDOCOUNT)
pvalues   = np.array([stats.ttest_ind(log2_norm[g, 3:], log2_norm[g, :3]).pvalue
                       for g in range(N_GENES)])

def bh_correction(pv):
    n = len(pv); order = np.argsort(pv); ranks = np.empty_like(order)
    ranks[order] = np.arange(1, n+1)
    adj = np.minimum(1.0, pv * n / ranks)
    a = adj[order]
    for i in range(n-2, -1, -1): a[i] = min(a[i], a[i+1])
    adj[order] = a
    return adj

padj      = bh_correction(pvalues)
sig_mask  = (padj < 0.05) & (np.abs(log2fc) > 1)
top50_idx = np.argsort(padj)[:50]

In [ ]:
# Publication rcParams (Module 18 NB07 standard)
plt.rcParams.update({
    'font.size': 9, 'axes.titlesize': 10, 'axes.labelsize': 9,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.spines.top': False, 'axes.spines.right': False,
    'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# Panel A: Heatmap of top 50 DE genes
ax = axes[0, 0]
top_norm = log2_norm[top50_idx, :]  # (50, 6)
# Z-score across samples
zmean = top_norm.mean(axis=1, keepdims=True)
zstd  = top_norm.std(axis=1, keepdims=True) + 1e-9
zscores = (top_norm - zmean) / zstd
im = ax.imshow(zscores, aspect='auto', cmap='RdBu_r', vmin=-2.5, vmax=2.5)
plt.colorbar(im, ax=ax, shrink=0.8, label='Row Z-score')
ax.set_xticks(range(6))
ax.set_xticklabels(['ctrl1','ctrl2','ctrl3','trt1','trt2','trt3'], fontsize=8, rotation=45, ha='right')
ax.set_ylabel('Top 50 DE genes (by adjusted p-value)')
ax.set_yticks([])
ax.set_title('A. Expression heatmap\n(top 50 DE genes, row-standardized)')
ax.text(-0.12, 1.05, 'A', transform=ax.transAxes, fontsize=12, fontweight='bold')
ax.axvline(2.5, color='white', lw=2)

# Panel B: GO enrichment simulation
ax = axes[0, 1]
# Assign random GO terms
go_terms = [f'GO:{1000+i}' for i in range(50)]
gene_go   = rng.choice(go_terms, N_GENES)  # 1 GO term per gene
de_genes  = de_idx
go_enrichment = []
for go in go_terms[:15]:
    n_go_total = (gene_go == go).sum()
    n_go_de    = (gene_go[de_genes] == go).sum()
    n_not_go_de   = N_DE - n_go_de
    n_go_notde    = n_go_total - n_go_de
    n_notgo_notde = N_GENES - N_DE - n_go_notde
    _, pv = stats.fisher_exact([[n_go_de, n_not_go_de], [n_go_notde, n_notgo_notde]])
    fold_enr = (n_go_de / max(N_DE, 1)) / (n_go_total / max(N_GENES, 1))
    go_enrichment.append({'term': go, 'pvalue': pv, 'fold_enrichment': fold_enr, 'n_de': n_go_de})
go_df = pd.DataFrame(go_enrichment).sort_values('pvalue').head(10)
colors_go = plt.cm.RdYlBu_r(np.linspace(0.2, 0.9, len(go_df)))
bars = ax.barh(range(len(go_df)), -np.log10(go_df['pvalue'].clip(1e-10)),
                color=colors_go, edgecolor='black', alpha=0.8)
ax.set_yticks(range(len(go_df)))
ax.set_yticklabels(go_df['term'].values, fontsize=8)
ax.set_xlabel('-log₁₀(Fisher exact p-value)')
ax.set_title('B. GO term enrichment\n(top 10 terms, simulated annotations)')
ax.text(-0.12, 1.05, 'B', transform=ax.transAxes, fontsize=12, fontweight='bold')

# Panel C: Violin plots for top 3 DE genes
ax = axes[1, 0]
top3_idx = np.argsort(padj)[:3]
cond_colors = ['#4e79a7', '#e15759']
for i, gidx in enumerate(top3_idx):
    for j, (start, label, color) in enumerate([(0,'ctrl','#4e79a7'),(3,'trt','#e15759')]):
        vals = log2_norm[gidx, start:start+3]
        parts = ax.violinplot(vals, positions=[i*3 + j], widths=0.8)
        for pc in parts['bodies']: pc.set_facecolor(color); pc.set_alpha(0.7)
        for k in ['cmins','cmaxes','cbars']: parts[k].set_color(color)
        ax.scatter(np.full(3, i*3 + j) + rng.uniform(-0.1, 0.1, 3), vals,
                    color='black', s=20, zorder=3)
ax.set_xticks([0.5, 3.5, 6.5])
ax.set_xticklabels([f'GENE_{top3_idx[k]:04d}' for k in range(3)], fontsize=8)
ax.set_ylabel('log₂(Normalized CPM + 1)')
ax.set_title('C. Top 3 DE genes: ctrl vs treated')
import matplotlib.patches as mpatches
ax.legend(handles=[mpatches.Patch(color='#4e79a7', label='Control'),
                     mpatches.Patch(color='#e15759', label='Treated')], fontsize=8)
ax.text(-0.12, 1.05, 'C', transform=ax.transAxes, fontsize=12, fontweight='bold')

# Panel D: Results text
ax = axes[1, 1]
ax.axis('off')
sig_count = sig_mask.sum()
tp = (sig_mask & np.isin(np.arange(N_GENES), de_idx)).sum()
results_text = (
    f'Results paragraph:\n\n'
    f'(Figure 1) Differential expression analysis identified {sig_count} '
    f'significantly DE genes (FDR < 0.05, |log₂FC| > 1) between treated '
    f'and control cells. Of the 300 injected DE genes, {tp} ({tp/300:.0%}) '
    f'were recovered (recall = {tp/300:.3f}). The MA plot (Figure 1B) shows '
    f'that DE genes are distributed across all expression levels. GO term '
    f'enrichment analysis (Figure 1B, right) identified {len(go_df)} nominally '
    f'enriched terms (Fisher exact p < 0.05); the top term ({go_df.iloc[0]["term"]}) '
    f'showed {go_df.iloc[0]["fold_enrichment"]:.1f}-fold enrichment.'
)
ax.text(0.05, 0.95, results_text, transform=ax.transAxes, fontsize=9,
          va='top', wrap=True,
          bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', edgecolor='grey'))
ax.set_title('D. Draft Results paragraph')
ax.text(-0.12, 1.05, 'D', transform=ax.transAxes, fontsize=12, fontweight='bold')

plt.suptitle('Module 20 CP01: Visualization and Biological Interpretation',
              fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('cp01_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 12 — Reflection

> *[Write a revised Results paragraph using the full statistical reporting
> standard from Module 18 NB05. Include: figure reference before the claim,
> exact numbers, confidence intervals, statistical test and n, and a
> one-sentence limitation.]*

---
*Next: `05_cp02_problem_setup_baselines.ipynb`*